In [ ]:
import torch 
import torch.optim as optim

In [ ]:
# temperature data in celcius
t_c = [0.5, 14.0, 15.0, 28.0, 11.0, 8.0, 3.0, -4.0, 6.0, 13.0, 21.0]

# temperature in unknown units
t_u = [35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4]

t_c = torch.tensor(t_c)
t_u = torch.tensor(t_u)

params = torch.tensor([1.0,1.0,0.0], requires_grad = True)
learning_rate = 1e-2
optimize = optim.SGD([params], lr = learning_rate)

def model(t_u, w2, w1, b):
    return w2 * t_u ** 2 + w1 * t_u ** 2 + b

def loss_fn(t_p, t_c):
    squared_diffs = (t_p-t_c)**2
    return squared_diffs.mean()

In [ ]:
def training_loop(n_epochs, optimizer, params, 
                  train_t_u, val_t_u,
                  train_t_c, val_t_c):
    for epoch in range(1, n_epochs+1):
        # get model predictions on training set
        train_t_p = model(train_t_u, *params)

        # evaluate loss on training set
        train_loss = loss_fn(train_t_p, train_t_c)

        # use context manager to avoid overhead of computational
        # graph when simply evaluating functions and not requiring
        # them in the calculation of gradients
        with torch.no_grad():
            # get model predictions on validation set
            val_t_p = model(val_t_u, *params)

            # evaluate loss on validation set
            val_loss = loss_fn(val_t_p, val_t_c)

            # ensure the gradient is not used
            assert val_loss.requires_grad == False
            
        # ensure zero grad
        optimizer.zero_grad()

        # calculate gradients on training data set
        train_loss.backward()

        # update parameters
        optimizer.step()

        # print results
        if epoch % 500 == 0:
            print(f"Epoch {epoch}, Training Loss {train_loss.item():.4f},"
                  f" Validation Loss {val_loss.item():.4f}")
    return params

In [ ]:
n_samples = t_u.shape[0]
# 20% goes to validation
n_val = int(0.2 * n_samples)

# generate 0 through n_samples-1, randomly permutated
shuffled_indices = torch.randperm(n_samples)

# train on all but the last 20%
train_indices = shuffled_indices[:-n_val]

# validate on the final 20%
val_indices = shuffled_indices[-n_val:]

train_indices, val_indices

In [ ]:
# make the train and test data sets

train_t_u = t_u[train_indices]
train_t_c = t_c[train_indices]

val_t_u = t_u[val_indices]
val_t_c = t_c[val_indices]

# normalize
train_t_un  = train_t_u * 0.1
val_t_un = val_t_u * 0.1

In [ ]:
params = torch.tensor([1.0,1.0,0.0], requires_grad = True)
learning_rate = 1e-1
optimizer = optim.Adam([params], lr = learning_rate)

opt_params = training_loop(
    n_epochs = 3000, 
    optimizer = optimizer, 
    params = params, 
    train_t_u = train_t_un,
    val_t_u = val_t_un, 
    train_t_c = train_t_c,
    val_t_c = val_t_c
)

In [ ]:
opt_params

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.scatter(t_u, t_c, label = "Data")

t_u_space = np.linspace(t_u.min().item(), t_u.max().item(),100)
t_un_space = torch.from_numpy(t_u_space*0.1)
with torch.no_grad():
    plt.plot(t_u_space, model(t_un_space,*opt_params))